# Mapped AcroForm Branch — First-time form, end-to-end

This notebook walks through the editable-form (AcroForm) workflow on a single input PDF as if the form has never been seen before. Every intermediate artifact is written to `output/` so the steps are inspectable.

**Python:** tested on **3.12.3**, requires **>= 3.10** (uses `X | Y` union syntax in inlined helpers).

**Standalone behavior:** every algorithmic step is inlined here so the partner can read the actual implementation. The only `claims_parser` imports are pydantic data models (so the JSON artifacts stay typed) and the Azure config loader.

Pipeline:

1. **Detect** the PDF is an AcroForm.
2. **Extract** the widget catalog (pymupdf).
3. **OCR** the form with Azure Document Intelligence.
4. **Fingerprint** the widget catalog (key for the reuse cache).
5. **Map** widgets → labels with the vision LLM.
6. **Build** the generic `FormSchema` from the mapping.
7. **Persist** the mapping under its fingerprint so a future run on the same blank form can skip steps 3, 5, 6.
8. **Fill** the schema with plausible dummy values (Agent 2).
9. **Write** the filled values into the AcroForm widgets.
10. **Summary**.

Change `INPUT_PDF` in cell 0 to demo a different form.

## 0. Setup

All paths are relative to the repo root. The cell below walks up the filesystem until it finds `claims_parser/`, then `chdir`s there so the `mappings/` cache directory resolves correctly. Secrets (Azure + OpenAI keys) live in `parser.env`.

In [ ]:
import os, sys
from pathlib import Path

assert sys.version_info >= (3, 10), "This notebook requires Python 3.10+"

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "claims_parser").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)
print("Python   :", sys.version.split()[0])

INPUT_PDF  = Path("input/ANTHEM_NV_CAID_ClaimsAppealsForm.pdf")  # change me
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

STEM = INPUT_PDF.stem
WIDGETS_JSON  = OUTPUT_DIR / f"{STEM}.widgets.json"
CONTEXT_JSON  = OUTPUT_DIR / f"{STEM}.context.json"
MAPPING_JSON  = OUTPUT_DIR / f"{STEM}.mapping.json"
SCHEMA_JSON   = OUTPUT_DIR / f"{STEM}.schema.json"
FILLED_JSON   = OUTPUT_DIR / f"{STEM}.filled.json"
FILLED_PDF    = OUTPUT_DIR / f"{STEM}.filled.pdf"
REVIEW_JSON   = OUTPUT_DIR / f"{STEM}.review.json"

print("Input :", INPUT_PDF)
print("Output:", OUTPUT_DIR.resolve())

### Tech stack used by this notebook

Every third-party library the workflow touches is imported here so the stack is visible up front.

| Library | Role |
|---|---|
| `pymupdf` (`fitz`) | Read AcroForm widgets, write filled values into the PDF |
| `azure-ai-documentintelligence` | OCR + layout (lines, KVPs, tables, selection marks) |
| `openai` | Vision-LLM widget→label mapping (Step 5) and dummy-value filler (Step 8), model `gpt-5-mini` |
| `pydantic` | Typed schemas for every intermediate JSON artifact |
| `python-dotenv` | Loads Azure + OpenAI keys from `parser.env` |

In [ ]:
import json, hashlib, base64, re
from datetime import datetime, timezone
from collections import Counter
from typing import Optional

import pymupdf                              # PDF read/write, AcroForm widgets
import openai                               # vision + filler LLM calls
from openai import OpenAI
import pydantic                             # typed schemas
import dotenv                               # loads parser.env
import azure.ai.documentintelligence as azure_di
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentAnalysisFeature
from azure.core.credentials import AzureKeyCredential

# Pydantic models from the project — kept as imports so JSON artifacts stay typed.
from claims_parser.widget_models  import Widget, WidgetCatalog, WidgetRect
from claims_parser.models         import (
    DocumentContext, PageMetadata, TextLine, KVP, Table, TableCell, SelectionMark,
)
from claims_parser.mapping_models import WidgetBinding, WidgetMapping, _LLMChunkResponse
from claims_parser.schema_models  import FormSchema, FormField, FieldType
from claims_parser.filler_models  import FilledFormSchema
from claims_parser.review_models  import ReviewItem, ReviewReport
from claims_parser.mapping_cache_models import CachedMapping, CacheIndex, CacheIndexEntry

# Load secrets from parser.env
dotenv.load_dotenv(REPO_ROOT / "parser.env")
AZURE_ENDPOINT = os.environ["AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT"]
AZURE_KEY      = os.environ["AZURE_DOCUMENT_INTELLIGENCE_KEY"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

print(f"pymupdf                       : {pymupdf.__version__}")
print(f"openai                        : {openai.__version__}")
print(f"pydantic                      : {pydantic.VERSION}")
print(f"python-dotenv                 : {dotenv.__version__}")
print(f"azure-ai-documentintelligence : {getattr(azure_di, '__version__', 'installed')}")
print("Azure endpoint                :", AZURE_ENDPOINT)

## 1. Detect — is this PDF editable?

Open the PDF with pymupdf and check whether any page has AcroForm widgets. Only AcroForm PDFs go through this branch.

In [ ]:
def is_acroform_pdf(pdf_path: Path) -> bool:
    doc = pymupdf.open(pdf_path)
    try:
        for page in doc:
            if any(True for _ in page.widgets()):
                return True
        return False
    finally:
        doc.close()

print("is_acroform_pdf:", is_acroform_pdf(INPUT_PDF))
print("branch         :", "acroform" if is_acroform_pdf(INPUT_PDF) else "scanned")

## 2. Extract the widget catalog

Pure pymupdf — no LLM, no Azure. Iterate every page, every widget, capture field name, type (text/checkbox/radio/dropdown), page, rect, choice values, on-value, and PDF xref. Radio-group options are emitted as separate widgets — that's how PDF AcroForms model them.

In [ ]:
WIDGET_TYPE_MAP = {
    "text":        "text",
    "checkbox":    "checkbox",
    "radiobutton": "radio",
    "listbox":     "choice",
    "combobox":    "choice",
    "signature":   "signature",
    "pushbutton":  "button",
}

def _on_value(w) -> Optional[str]:
    try:
        v = w.on_state()
    except Exception:
        v = None
    return v or None

def extract_widget_catalog(pdf_path: Path) -> WidgetCatalog:
    doc = pymupdf.open(pdf_path)
    try:
        page_sizes, widgets = [], []
        for page_idx, page in enumerate(doc, start=1):
            page_sizes.append((page.rect.width, page.rect.height))
            for w in page.widgets() or []:
                wt = WIDGET_TYPE_MAP.get((w.field_type_string or "").lower(), "text")
                r = w.rect
                widgets.append(Widget(
                    field_name=w.field_name or f"_unnamed_{w.xref}",
                    widget_type=wt,
                    rect=WidgetRect(page=page_idx, x0=r.x0, y0=r.y0, x1=r.x1, y1=r.y1),
                    xref=w.xref,
                    tu_label=(w.field_label or None),
                    choice_values=(list(w.choice_values) if getattr(w, "choice_values", None) else None),
                    on_value=_on_value(w) if wt in ("checkbox", "radio") else None,
                ))
        return WidgetCatalog(
            file_name=pdf_path.name,
            page_count=doc.page_count,
            page_sizes_pt=page_sizes,
            widgets=widgets,
        )
    finally:
        doc.close()

catalog = extract_widget_catalog(INPUT_PDF)
WIDGETS_JSON.write_text(catalog.model_dump_json(indent=2))

print(f"Pages   : {catalog.page_count}")
print(f"Widgets : {len(catalog.widgets)}")
for t, c in Counter(w.widget_type for w in catalog.widgets).most_common():
    print(f"  {t:10s} {c}")
print(f"Saved   : {WIDGETS_JSON}")

## 3. OCR the form with Azure Document Intelligence

Calls Azure's `prebuilt-layout` model with the key-value-pairs feature. F0 tier caps each request at 2 pages, so we chunk the PDF and stitch the results back together with page-number offsets.

In [ ]:
AZURE_MODEL = "prebuilt-layout"
AZURE_CHUNK_PAGES = 2

def _polygon_to_list(polygon):
    if not polygon:
        return []
    first = polygon[0]
    if hasattr(first, "x") and hasattr(first, "y"):
        return [c for pt in polygon for c in (pt.x, pt.y)]
    return list(polygon)

def _first_region(regions):
    if not regions:
        return None, None
    r = regions[0]
    return getattr(r, "page_number", None), (_polygon_to_list(getattr(r, "polygon", None)) or None)

def _chunk_pdf_bytes(src_path: Path, start: int, end_inclusive: int) -> bytes:
    sub, src = pymupdf.open(), pymupdf.open(src_path)
    try:
        sub.insert_pdf(src, from_page=start, to_page=end_inclusive)
        return sub.tobytes()
    finally:
        sub.close(); src.close()

def _ingest(result, page_offset: int):
    pages, lines, marks = [], [], []
    for page in result.pages or []:
        actual = (page.page_number or 1) + page_offset
        pages.append(PageMetadata(page_number=actual, width=page.width or 0.0,
                                  height=page.height or 0.0, unit=page.unit or "inch",
                                  line_count=len(page.lines or [])))
        for ln in page.lines or []:
            lines.append(TextLine(text=ln.content, page=actual,
                                  polygon=_polygon_to_list(ln.polygon)))
        for m in getattr(page, "selection_marks", None) or []:
            marks.append(SelectionMark(page=actual,
                                       state=getattr(m, "state", "unselected") or "unselected",
                                       polygon=_polygon_to_list(m.polygon),
                                       confidence=getattr(m, "confidence", None)))
    tables = []
    for tbl in result.tables or []:
        tpage, _ = _first_region(getattr(tbl, "bounding_regions", None))
        cells = []
        for c in tbl.cells:
            cpage, cpoly = _first_region(getattr(c, "bounding_regions", None))
            cells.append(TableCell(row=c.row_index, column=c.column_index, text=c.content or "",
                                   page=(cpage or tpage or 1) + page_offset, polygon=cpoly,
                                   row_span=c.row_span or 1, column_span=c.column_span or 1,
                                   kind=getattr(c, "kind", None)))
        tables.append(Table(page=(tpage or 1) + page_offset,
                            row_count=tbl.row_count, column_count=tbl.column_count, cells=cells))
    kvps = []
    for kvp in result.key_value_pairs or []:
        kpage, kpoly = _first_region(getattr(kvp.key, "bounding_regions", None))
        vpage, vpoly, vtext = None, None, None
        if kvp.value is not None:
            vpage, vpoly = _first_region(getattr(kvp.value, "bounding_regions", None))
            vtext = kvp.value.content
        kvps.append(KVP(key_text=kvp.key.content,
                        key_page=(kpage + page_offset) if kpage else None, key_polygon=kpoly,
                        value_text=vtext,
                        value_page=(vpage + page_offset) if vpage else None, value_polygon=vpoly,
                        confidence=getattr(kvp, "confidence", None)))
    return pages, lines, tables, kvps, marks, result.content or ""

def extract_document_context(pdf_path: Path) -> DocumentContext:
    client = DocumentIntelligenceClient(endpoint=AZURE_ENDPOINT,
                                        credential=AzureKeyCredential(AZURE_KEY))
    total = pymupdf.open(pdf_path).page_count
    all_pages, all_lines, all_tables, all_kvps, all_marks, parts = [], [], [], [], [], []
    for start in range(0, total, AZURE_CHUNK_PAGES):
        end = min(start + AZURE_CHUNK_PAGES - 1, total - 1)
        print(f"  Azure chunk: pages {start+1}-{end+1}")
        poller = client.begin_analyze_document(
            model_id=AZURE_MODEL,
            body=_chunk_pdf_bytes(pdf_path, start, end),
            features=[DocumentAnalysisFeature.KEY_VALUE_PAIRS],
            content_type="application/octet-stream",
        )
        p, l, t, k, m, c = _ingest(poller.result(), start)
        all_pages += p; all_lines += l; all_tables += t
        all_kvps  += k; all_marks += m; parts.append(c)
    return DocumentContext(
        file_name=pdf_path.name, file_path=str(pdf_path), model_id=AZURE_MODEL,
        full_text="\n".join(parts), pages=all_pages, lines=all_lines,
        tables=all_tables, key_value_pairs=all_kvps, selection_marks=all_marks,
    )

doc_ctx = extract_document_context(INPUT_PDF)
CONTEXT_JSON.write_text(doc_ctx.model_dump_json(indent=2))

print(f"Pages           : {len(doc_ctx.pages)}")
print(f"Lines           : {len(doc_ctx.lines)}")
print(f"Key-value pairs : {len(doc_ctx.key_value_pairs)}")
print(f"Tables          : {len(doc_ctx.tables)}")
print(f"Selection marks : {len(doc_ctx.selection_marks)}")
print(f"Saved           : {CONTEXT_JSON}")

## 4. Compute the form fingerprint

SHA-256 over a canonical JSON of the widget catalog: widgets sorted by `(page, field_name, on_value, x0, y0)`; rects rounded to ~0.5pt (`round(v / 1.0) * 1.0` collapses ±0.5pt jitter into the same bucket); `xref` and `tu_label` excluded (not stable across PDF copies). Two users filling the same blank form produce the same fingerprint.

In [ ]:
FINGERPRINT_VERSION = 1
RECT_BUCKET_PT = 1.0

def _bucket(v: float) -> float:
    return round(v / RECT_BUCKET_PT) * RECT_BUCKET_PT

def compute_form_fingerprint(catalog: WidgetCatalog) -> str:
    widgets_sorted = sorted(
        catalog.widgets,
        key=lambda w: (w.rect.page, w.field_name, w.on_value or "",
                       _bucket(w.rect.x0), _bucket(w.rect.y0)),
    )
    payload = {
        "v": FINGERPRINT_VERSION,
        "page_count": catalog.page_count,
        "page_sizes_pt": [[_bucket(w), _bucket(h)] for (w, h) in catalog.page_sizes_pt],
        "widgets": [
            {
                "field_name": w.field_name,
                "widget_type": w.widget_type,
                "page": w.rect.page,
                "on_value": w.on_value,
                "choice_values": list(w.choice_values) if w.choice_values else None,
                "rect": [_bucket(w.rect.x0), _bucket(w.rect.y0),
                         _bucket(w.rect.x1), _bucket(w.rect.y1)],
            }
            for w in widgets_sorted
        ],
    }
    blob = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(blob).hexdigest()

fingerprint = compute_form_fingerprint(catalog)
print("Fingerprint:", fingerprint)

## 5. Map widgets → labels (vision LLM)

The expensive step. For each chunk of ≤4 pages we render a high-DPI PNG, pack the widget rects + Azure OCR hints (KVPs, text lines, table cells, selection marks) into a JSON payload, and call `gpt-5-mini` with a strict system prompt. `client.chat.completions.parse(response_format=_LLMChunkResponse)` gives us a structured-output guarantee. Each widget ends up either in `bindings` (with a label and confidence) or in `unmapped_widget_field_names`.

In [ ]:
MAPPER_MODEL          = "gpt-5-mini"
MAPPER_CHUNK_PAGES    = 4
MAPPER_RENDER_DPI     = 144
MAPPER_SPATIAL_TOL_PT = 100.0

MAPPER_SYSTEM_PROMPT = """You bind every fillable AcroForm widget on a PDF page to the printed label that
names it on the form.

INPUTS PER CALL
- One or more page images: the authoritative visual rendering of the pages.
- A DI hint set: labels the layout OCR detected on these pages — KVPs,
  text lines, table cells, selection marks. Each carries a polygon in PDF
  points.
- A widget catalog: every input field on these pages with its rect in PDF
  points, widget_type, opaque PDF field_name, optional author-set tu_label,
  on_value, choice_values, and xref.

COORDINATE SYSTEM
- Both page images and PDF polygons use top-left origin with y growing
  downward. All measurements are in PDF points (1 pt = 1/72 inch).

OUTPUT CONTRACT
- Emit one WidgetBinding per widget OR list the widget's field_name in
  `unmapped_widget_field_names`. Every widget MUST appear in exactly one
  of the two outputs.
- label_text: the printed label, copied verbatim.
- label_source: "kvp", "text_line", "table_cell", or "image_only".
- label_polygon: copy verbatim from the DI hint. Null ONLY when image_only.
- label_page MUST equal the widget's rect page. Never bind across pages.
- semantic_field_id: lowercase snake_case derived from the label. Radio
  siblings sharing a label share this id.
- option_label: for radio siblings, the choice text adjacent to THIS widget
  (e.g. "Yes" / "No"). Otherwise null.
- confidence: 0..1. reasoning: one short sentence.

HARD RULES
1. Every widget MUST end in bindings or unmapped_widget_field_names.
2. Never paraphrase label_text. Copy verbatim.
3. Never invent a label that is not present on THIS form's image.
4. If no plausible printed label is visible near a widget, mark it unmapped.
"""

def _render_page_b64(doc, page_idx: int, dpi: int) -> str:
    return base64.b64encode(doc[page_idx].get_pixmap(dpi=dpi).tobytes("png")).decode("ascii")

def _di_for_pages(ctx: DocumentContext, pages: set[int]) -> dict:
    return {
        "key_value_pairs": [k.model_dump() for k in ctx.key_value_pairs if k.key_page in pages],
        "text_lines":      [l.model_dump() for l in ctx.lines if l.page in pages],
        "table_cells":     [c.model_dump() for t in ctx.tables for c in t.cells
                            if c.page in pages and c.text.strip()],
        "selection_marks": [m.model_dump() for m in ctx.selection_marks if m.page in pages],
    }

def _rect_label_distance(rect, polygon: list[float]) -> float:
    xs, ys = polygon[0::2], polygon[1::2]
    lx0, lx1 = min(xs), max(xs); ly0, ly1 = min(ys), max(ys)
    dx = max(rect.x0 - lx1, lx0 - rect.x1, 0.0)
    dy = max(rect.y0 - ly1, ly0 - rect.y1, 0.0)
    return (dx * dx + dy * dy) ** 0.5

def _validate_chunk(parsed, chunk_widgets):
    by_xref = {w.xref: w for w in chunk_widgets}
    by_name = {w.field_name: w for w in chunk_widgets}
    seen, valid = set(), []
    unmapped = list(parsed.unmapped_widget_field_names)
    for b in parsed.bindings:
        w = by_xref.get(b.widget_xref) or by_name.get(b.widget_field_name)
        if w is None or w.xref in seen: continue
        if b.label_page != w.rect.page:
            unmapped.append(w.field_name); seen.add(w.xref); continue
        if b.label_source != "image_only" and b.label_polygon:
            if _rect_label_distance(w.rect, b.label_polygon) > MAPPER_SPATIAL_TOL_PT:
                b = b.model_copy(update={"confidence": max(0.0, b.confidence - 0.2)})
        b = b.model_copy(update={
            "widget_xref": w.xref, "widget_field_name": w.field_name,
            "on_value": w.on_value if w.widget_type in ("checkbox", "radio") else None,
        })
        valid.append(b); seen.add(w.xref)
    for w in chunk_widgets:
        if w.xref not in seen and w.field_name not in unmapped:
            unmapped.append(w.field_name)
    return valid, list(dict.fromkeys(unmapped))

def map_widgets(pdf_path: Path, ctx: DocumentContext, catalog: WidgetCatalog,
                model: str = MAPPER_MODEL) -> WidgetMapping:
    client = OpenAI(api_key=OPENAI_API_KEY)
    widgets_by_page = {}
    for w in catalog.widgets:
        widgets_by_page.setdefault(w.rect.page, []).append(w)

    all_bindings, all_unmapped, chunks = [], [], 0
    doc = pymupdf.open(pdf_path)
    try:
        for start in range(1, catalog.page_count + 1, MAPPER_CHUNK_PAGES):
            pages = list(range(start, min(start + MAPPER_CHUNK_PAGES, catalog.page_count + 1)))
            chunk_widgets = [w for p in pages for w in widgets_by_page.get(p, [])]
            if not chunk_widgets: continue
            chunks += 1

            text_payload = {
                "pages": pages,
                "page_sizes_pt": {p: catalog.page_sizes_pt[p - 1] for p in pages},
                "widgets": [w.model_dump() for w in chunk_widgets],
                "di_hints": _di_for_pages(ctx, set(pages)),
            }
            content = [{"type": "text", "text": json.dumps(text_payload, separators=(",", ":"))}]
            for p in pages:
                png = _render_page_b64(doc, p - 1, MAPPER_RENDER_DPI)
                content.append({"type": "image_url",
                                "image_url": {"url": f"data:image/png;base64,{png}",
                                              "detail": "high"}})

            print(f"  LLM call: pages {pages[0]}-{pages[-1]} ({len(chunk_widgets)} widgets)")
            response = client.chat.completions.parse(
                model=model,
                messages=[{"role": "system", "content": MAPPER_SYSTEM_PROMPT},
                          {"role": "user", "content": content}],
                response_format=_LLMChunkResponse,
            )
            parsed = response.choices[0].message.parsed
            if parsed is None:
                all_unmapped.extend(w.field_name for w in chunk_widgets); continue
            v, u = _validate_chunk(parsed, chunk_widgets)
            all_bindings += v; all_unmapped += u
    finally:
        doc.close()

    return WidgetMapping(file_name=pdf_path.name, model=model, chunk_count=chunks,
                         bindings=all_bindings,
                         unmapped_widget_field_names=list(dict.fromkeys(all_unmapped)))

print(f"Mapping with {MAPPER_MODEL}...")
mapping = map_widgets(INPUT_PDF, doc_ctx, catalog)
MAPPING_JSON.write_text(mapping.model_dump_json(indent=2))

print(f"Bindings  : {len(mapping.bindings)}")
print(f"Unmapped  : {len(mapping.unmapped_widget_field_names)}")
print(f"Chunks    : {mapping.chunk_count}")
print(f"Saved     : {MAPPING_JSON}")

## 6. Build the FormSchema

Turn the widget→label bindings into the generic `FormSchema` shape Agent 2 consumes. Group bindings by `semantic_field_id`. Radio groups collapse multiple bindings into one `FormField` with an `options` list; everything else becomes one field per widget. `field_type` is derived from the widget type and rect height (tall text widgets become `long_text`).

In [ ]:
LONG_TEXT_MIN_HEIGHT_PT = 30.0

def _snake(label: str) -> str:
    s = re.sub(r"[^a-zA-Z0-9]+", "_", label.strip().lower()).strip("_")
    return s or "field"

def _uniqify(field_id: str, taken: set) -> str:
    if field_id not in taken:
        taken.add(field_id); return field_id
    i = 2
    while f"{field_id}_{i}" in taken:
        i += 1
    out = f"{field_id}_{i}"; taken.add(out); return out

def _type_and_options(w: Widget, b: WidgetBinding):
    if w.widget_type == "checkbox":  return "checkbox", [b.label_text]
    if w.widget_type == "choice":    return "radio_group", list(w.choice_values or [])
    if w.widget_type == "signature": return "signature", None
    height = w.rect.y1 - w.rect.y0
    return ("long_text" if height > LONG_TEXT_MIN_HEIGHT_PT else "text"), None

def build_schema_from_mapping(catalog: WidgetCatalog, mapping: WidgetMapping) -> FormSchema:
    by_xref = {w.xref: w for w in catalog.widgets}
    groups: dict[str, list[WidgetBinding]] = {}
    for b in mapping.bindings:
        groups.setdefault(b.semantic_field_id, []).append(b)

    taken, fields = set(), []
    for raw_id, group in groups.items():
        widgets = [by_xref.get(b.widget_xref) for b in group]
        widgets = [w for w in widgets if w is not None]
        if not widgets: continue
        types = {w.widget_type for w in widgets}

        if types == {"radio"} and len(group) > 1:
            label = group[0].label_text
            options = [b.option_label or b.on_value or "Option" for b in group]
            fields.append(FormField(
                field_id=_uniqify(_snake(raw_id), taken), label=label, section=None,
                field_type="radio_group", options=options, format_hint=None,
                required=False, source_text=label, widget_bindings=group,
            ))
        else:
            for b, w in zip(group, widgets):
                ftype, options = _type_and_options(w, b)
                fields.append(FormField(
                    field_id=_uniqify(_snake(b.semantic_field_id), taken),
                    label=b.label_text, section=None, field_type=ftype, options=options,
                    format_hint=None, required=False, source_text=b.label_text,
                    widget_bindings=[b],
                ))
    return FormSchema(form_title=None, issuer=None, sections=[], fields=fields)

schema = build_schema_from_mapping(catalog, mapping)
SCHEMA_JSON.write_text(schema.model_dump_json(indent=2))

print(f"Fields   : {len(schema.fields)}")
print(f"Sections : {len(schema.sections)}")
print(f"Saved    : {SCHEMA_JSON}")

## 7. Persist the mapping under its fingerprint

Write `mappings/<fingerprint>.{mapping,schema,cached}.json` plus an index entry. The next time anyone runs the workflow on this blank form, the fingerprint matches and Steps 3, 5, and 6 are skipped.

In [ ]:
CACHE_DIR = REPO_ROOT / "mappings"

def _atomic_write(path: Path, content: str) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(content)
    os.replace(tmp, path)

def store_cached_mapping(fingerprint: str, mapping: WidgetMapping, schema: FormSchema,
                         source_pdf_basename: str) -> None:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    created = datetime.now(timezone.utc).isoformat(timespec="seconds")
    cached = CachedMapping(form_fingerprint=fingerprint, source_pdf_basename=source_pdf_basename,
                           created_at=created, mapping=mapping, form_schema=schema)
    _atomic_write(CACHE_DIR / f"{fingerprint}.cached.json",  cached.model_dump_json(indent=2))
    _atomic_write(CACHE_DIR / f"{fingerprint}.mapping.json", mapping.model_dump_json(indent=2))
    _atomic_write(CACHE_DIR / f"{fingerprint}.schema.json",  schema.model_dump_json(indent=2))

    idx_path = CACHE_DIR / "index.json"
    index = CacheIndex.model_validate_json(idx_path.read_text()) if idx_path.exists() else CacheIndex(entries=[])
    index.entries = [e for e in index.entries if e.form_fingerprint != fingerprint]
    index.entries.append(CacheIndexEntry(form_fingerprint=fingerprint,
                                         source_pdf_basename=source_pdf_basename,
                                         created_at=created))
    _atomic_write(idx_path, index.model_dump_json(indent=2))

store_cached_mapping(fingerprint, mapping, schema, source_pdf_basename=INPUT_PDF.name)

print(f"Cache dir : {CACHE_DIR}")
for p in sorted(CACHE_DIR.glob(f"{fingerprint}*")):
    print(f"  {p.name}")

## 8. Fill the schema (Agent 2)

Strip the `widget_bindings` from the schema (Agent 2 doesn't need them), JSON-serialize, and ask `gpt-5-mini` for a plausible value per field. Structured-output via `response_format=FilledFormSchema` guarantees we get back the same shape. After the call we re-attach the bindings so Step 9 knows which widgets to write into.

In [ ]:
FILLER_MODEL = "gpt-5-mini"

FILLER_SYSTEM_PROMPT = """You receive a generic JSON schema describing every fillable input on a form
and must return the same schema with a plausible `value` filled in for each field.

Rules:
- Fill EVERY field with a plausible non-empty value. Even if the form text
  suggests a field only applies under some condition (e.g. "If yes, date:"),
  treat the condition as if it applies and produce a value.
- Produce values that are internally consistent across fields (same
  fictional individual; dates mutually plausible).
- Values must be fictional but realistic. Do not use real personal data.
- Respect field_type:
    text / long_text  -> a single non-empty string
    date              -> match format_hint if present, otherwise ISO YYYY-MM-DD
    time              -> match format_hint if present, otherwise HH:MM
    phone / email / address / number / currency / identifier / signature -> a single non-empty string
    checkbox          -> a non-empty list drawn ONLY from this field's options
    radio_group       -> exactly one string drawn ONLY from this field's options
- Never invent options that are not in the field's options list.
- Do not modify field_id, label, section, field_type, options, format_hint,
  required, or source_text. Only fill `value`.
"""

def fill_form_schema(schema: FormSchema, model: str = FILLER_MODEL) -> FilledFormSchema:
    client = OpenAI(api_key=OPENAI_API_KEY)
    schema_dict = schema.model_dump()
    for f in schema_dict.get("fields", []):
        f.pop("widget_bindings", None)
    response = client.chat.completions.parse(
        model=model,
        messages=[{"role": "system", "content": FILLER_SYSTEM_PROMPT},
                  {"role": "user",   "content": json.dumps(schema_dict, indent=2)}],
        response_format=FilledFormSchema,
    )
    filled = response.choices[0].message.parsed

    bindings_by_id = {f.field_id: f.widget_bindings for f in schema.fields}
    for ff in filled.fields:
        if bindings_by_id.get(ff.field_id) is not None:
            ff.widget_bindings = bindings_by_id[ff.field_id]
    return filled

filled = fill_form_schema(schema)
FILLED_JSON.write_text(filled.model_dump_json(indent=2))

n_filled = sum(1 for f in filled.fields if f.value not in (None, "", []))
print(f"Filled : {n_filled}/{len(filled.fields)}")
print(f"Saved  : {FILLED_JSON}")

print("\nSample values:")
for f in filled.fields[:8]:
    print(f"  {f.field_id:40s} {f.value!r}")

## 9. Write the filled PDF

For each filled field, look up its widget by `xref` on the relevant page and set `widget.field_value`. Radio groups get special handling: writing only the chosen sibling leaves stale `/AS` on the others (no dot rendered in viewers), so we explicitly set every other sibling to `"Off"` first, then write the chosen one last.

In [ ]:
LOW_CONFIDENCE_THRESHOLD = 0.5

def _find_widget(page, xref: int):
    for w in page.widgets() or []:
        if w.xref == xref:
            return w
    return None

def _set_text(widget, value) -> bool:
    try:
        widget.field_value = "" if value is None else str(value)
        widget.update(); return True
    except Exception:
        return False

def _set_button_on(widget, on_value) -> bool:
    try:
        widget.field_value = on_value if on_value else True
        widget.update(); return True
    except Exception:
        return False

def _norm(s) -> str:
    return (s or "").strip().lower()

def _select_radio_binding(bindings, value):
    target = _norm(value if isinstance(value, str) else "")
    if not target: return None
    for b in bindings:
        if _norm(b.option_label) == target or _norm(b.on_value) == target:
            return b
    for b in bindings:
        ol = _norm(b.option_label)
        if ol and (target in ol or ol in target):
            return b
    return None

def write_filled_pdf(pdf_path: Path, filled: FilledFormSchema,
                     output_path: Path, flatten: bool = False) -> ReviewReport:
    review = []
    doc = pymupdf.open(pdf_path)
    try:
        for field in filled.fields:
            bindings = field.widget_bindings or []
            if not bindings:
                review.append(ReviewItem(field_id=field.field_id, label=field.label,
                                         section=field.section, reason="widget_unmapped"))
                continue
            if field.value in (None, "", []):
                review.append(ReviewItem(field_id=field.field_id, label=field.label,
                                         section=field.section,
                                         page=bindings[0].label_page, reason="missing_value"))
                continue

            min_conf = min(b.confidence for b in bindings)
            if min_conf < LOW_CONFIDENCE_THRESHOLD:
                review.append(ReviewItem(field_id=field.field_id, label=field.label,
                                         section=field.section, page=bindings[0].label_page,
                                         reason="widget_low_confidence",
                                         details=f"min binding confidence={min_conf:.2f}"))

            # Radio group with multiple sibling widgets
            if field.field_type == "radio_group" and len(bindings) > 1:
                chosen = _select_radio_binding(bindings, field.value)
                if chosen is None:
                    review.append(ReviewItem(field_id=field.field_id, label=field.label,
                                             page=bindings[0].label_page, reason="unsettable_widget",
                                             details=f"no option binding matches value {field.value!r}"))
                    continue
                # Force every other sibling Off so viewers don't render a stale dot.
                for b in bindings:
                    if b.widget_xref == chosen.widget_xref: continue
                    sib = _find_widget(doc[b.label_page - 1], b.widget_xref)
                    if sib is not None:
                        try: sib.field_value = "Off"; sib.update()
                        except Exception: pass
                w = _find_widget(doc[chosen.label_page - 1], chosen.widget_xref)
                if w is None or not _set_button_on(w, chosen.on_value):
                    review.append(ReviewItem(field_id=field.field_id, label=field.label,
                                             page=chosen.label_page, reason="unsettable_widget"))
                continue

            # Single-widget field
            b = bindings[0]
            w = _find_widget(doc[b.label_page - 1], b.widget_xref)
            if w is None:
                review.append(ReviewItem(field_id=field.field_id, label=field.label,
                                         page=b.label_page, reason="widget_unmapped",
                                         details="widget xref not found at write time"))
                continue

            if field.field_type == "checkbox":
                if field.value:  # truthy = tick it
                    if not _set_button_on(w, b.on_value):
                        review.append(ReviewItem(field_id=field.field_id, label=field.label,
                                                 page=b.label_page, reason="unsettable_widget"))
                continue

            if not _set_text(w, field.value):
                review.append(ReviewItem(field_id=field.field_id, label=field.label,
                                         page=b.label_page, reason="unsettable_widget"))

        if flatten:
            try: doc.bake()
            except Exception: pass
        doc.save(output_path, incremental=False, deflate=True)
    finally:
        doc.close()
    return ReviewReport(items=review)

review = write_filled_pdf(INPUT_PDF, filled, FILLED_PDF, flatten=False)
REVIEW_JSON.write_text(review.model_dump_json(indent=2))

print(f"Filled PDF : {FILLED_PDF}")
print(f"Review     : {REVIEW_JSON}   ({len(review.items)} items)")
for reason, count in sorted(review.by_reason().items()):
    print(f"  {reason:24s} {count}")

## 10. Summary

Per-run artifacts live under `output/`. The reusable mapping lives under `mappings/` keyed by the form's fingerprint.

In [ ]:
print("Per-run artifacts (output/):")
for p in [WIDGETS_JSON, CONTEXT_JSON, MAPPING_JSON, SCHEMA_JSON,
          FILLED_JSON, FILLED_PDF, REVIEW_JSON]:
    mark = "OK" if p.exists() else "--"
    print(f"  [{mark}] {p}")

print(f"\nReusable cache entry:")
for p in sorted(CACHE_DIR.glob(f"{fingerprint}*")):
    print(f"  {p}")